# 🧠 1️⃣ ¿Qué es un CTE?

## Un CTE es una tabla temporal con nombre que existe solo durante la consulta.

Se crea con:

WITH nombre_cte AS (
    consulta
)
SELECT * FROM nombre_cte;

Es como decir:

## “Primero construyo esta tabla temporal… luego la uso.”

In [1]:
import duckdb
import pandas as pd
# Conectarse a DuckDB
con = duckdb.connect('../data/analytics.db')

con.execute("SHOW TABLES").fetchdf()


,name
0,customers
1,transactions


In [17]:
con.execute("DESCRIBE customers").fetchdf()

,column_name,column_type,null,key,default,extra
0,customer_id,INTEGER,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,city,VARCHAR,YES,None,None,None


In [3]:
# como imprimir columnas d la tabla
con.execute("DESCRIBE transactions").fetchdf()

,column_name,column_type,null,key,default,extra
0,transaction_id,INTEGER,YES,None,None,None
1,customer_id,INTEGER,YES,None,None,None
2,amount,DOUBLE,YES,None,None,None
3,date,DATE,YES,None,None,None


# <span style="color:red">SIN CTES ❌</span>

In [5]:
con.execute("""
SELECT customer_id, SUM(amount) AS total_gasto
FROM transactions
GROUP BY customer_id;
            
            """).fetchdf()


,customer_id,total_gasto
0,1,400.0
1,2,230.0
2,3,250.0
3,4,75.0
4,5,500.0
5,6,250.0
6,7,300.0
7,8,50.0


# <span style="color:green">Con CTES</span>

In [8]:
con.execute("""
            
WITH customer_totals AS ( select customer_id, SUM(amount) AS total_gasto
FROM transactions
GROUP BY customer_id )
            
select customer_id, total_gasto
from customer_totals
            

        """).fetchdf()

,customer_id,total_gasto
0,1,400.0
1,2,230.0
2,3,250.0
3,4,75.0
4,5,500.0
5,6,250.0
6,7,300.0
7,8,50.0




# 🟢 Nivel 1 — CTE básico

### 🎯 Ejercicio 1

Obtén el **total gastado por cada cliente** usando un CTE.

**Reglas**

* Usa `WITH`
* Usa `SUM`
* Usa `GROUP BY`


In [11]:
con.execute("""
with customer_totals as (
            select customer_id, SUM(amount) AS total_gasto
            from transactions
            group by customer_id
            )

select customer_id, total_gasto
from customer_totals
""").fetch_df()

,customer_id,total_gasto
0,1,400.0
1,2,230.0
2,3,250.0
3,4,75.0
4,5,500.0
5,6,250.0
6,7,300.0
7,8,50.0


### 🎯 Ejercicio 2

Obtén los **clientes que han hecho más de 5 transacciones**.

**Reglas**

* Usa CTE
* Usa `COUNT`
* Usa `GROUP BY`



In [16]:
con.execute("""
with count_transactions as (
    select customer_id, count(transaction_id) as num_transaction
    from transactions
    group by customer_id
)
select * from count_transactions
where num_transaction > 5;
""").fetch_df()

,customer_id,num_transaction



# 🟠 Nivel 3 — CTE + comparación (muy importante)

### 🎯 Ejercicio 5

Obtén los clientes cuyo gasto total sea mayor que el promedio de todos los clientes.

⚠️ Este es clásico de entrevistas.

In [24]:
con.execute("""
WITH 
    gastos_totales AS (
        SELECT 
            customer_id, 
            SUM(amount) AS gasto_total
        FROM transactions
        GROUP BY customer_id
    ),
    avg_gastos_totales AS (
        SELECT 
            AVG(gasto_total) AS promedio
        FROM gastos_totales
    )

SELECT customer_id  
FROM gastos_totales
WHERE gasto_total > (
    SELECT promedio 
    FROM avg_gastos_totales
);
""").fetch_df()

,customer_id
0,1
1,5
2,7


### 🎯 Ejercicio 8

Obtén el cliente que más dinero ha gastado.

**Pista mental**
CTE 1 → gasto por cliente
CTE 2 → máximo gasto
Consulta final → filtrar


In [25]:
con.execute("""
WITH 
    gastos_totales AS (
        SELECT 
            customer_id,
            SUM(amount) AS total_gasto
        FROM transactions
        GROUP BY customer_id
    ),
    max_gasto AS (
        SELECT 
            MAX(total_gasto) AS gasto_maximo
        FROM gastos_totales
    )

SELECT customer_id, total_gasto
FROM gastos_totales
WHERE total_gasto = (
    SELECT gasto_maximo 
    FROM max_gasto
);
""").fetch_df()

,customer_id,total_gasto
0,5,500.0
